# Lekce 11 - Protokol modelového kontextu (MCP)

Protokol **Model Context Protocol (MCP)** je otevřený standard, který umožňuje agentům dynamicky objevovat a používat nástroje, zdroje a datové zdroje během běhu. Místo že by byly nástroje pevně zakódovány v agentovi, MCP umožňuje agentům připojovat se k externím serverům, které na vyžádání zpřístupňují schopnosti.

V této lekci se naučíte:
- Co je MCP a proč je důležité pro agentní systémy
- Jak funguje architektura klient-server MCP
- Jak vytvářet agenty, kteří používají objevování nástrojů ve stylu MCP


## Nastavení

**Požadavky:**
- Projekt Microsoft Foundry s nasazeným modelem
- Pro autentizaci spusťte `az login`


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Co je Model Context Protocol (MCP)?

MCP definuje standardní způsob, jak mohou AI agenti objevovat a interagovat s externími nástroji a zdroji dat:

- **MCP Server**: Zpřístupňuje nástroje, zdroje a výzvy přes standardní protokol
- **MCP Client**: Běhové prostředí agenta, které se připojuje k serverům a objevuje dostupné schopnosti
- **Dynamické objevování**: Agentům nemusí být nástroje pevně zakódovány — objeví, co je dostupné za běhu

To je silné pro budování rozšiřitelných systémů agentů, kde lze přidávat nové schopnosti bez úprav kódu agenta.


## Jak MCP funguje

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agent (MCP klient) se připojí k MCP serveru
2. Server odpoví seznamem dostupných nástrojů a jejich schématy
3. Agent může během svého uvažování volat libovolný nalezený nástroj
4. Výsledky se vrací zpět stejným protokolem


## Simulace zjišťování nástrojů MCP

Protože skutečný server MCP vyžaduje běžící serverový proces, předvedeme vzor pomocí funkcí `@tool`, které simulují to, co by služba ubytování připojená k MCP poskytla.

Ve výrobním prostředí by tyto nástroje byly zjišťovány dynamicky ze serveru MCP místo toho, aby byly definovány lokálně.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Vytváření agenta s nástroji ve stylu MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP v produkci

V produkčním prostředí MCP umožňuje výkonné vzory:

- **Dynamické objevování nástrojů**: Agenti se připojují k MCP serverům a za běhu objevují nástroje
- **Oddělená architektura**: Poskytovatelé nástrojů se mohou aktualizovat nezávisle na agentovi
- **Sdílení napříč organizacemi**: Týmy mohou zpřístupnit funkce prostřednictvím MCP serverů, které může využít každý agent
- **Podpora Microsoft Agent Framework**: MAF má zabudovanou podporu klienta MCP prostřednictvím integrace `mcp`

Pro použití skutečného MCP serveru s MAF se připojíte pomocí `hosted_mcp_tool()` nebo integrace klienta MCP.

**Více informací:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

In this lesson, you learned:
- **MCP** is an open standard for dynamic tool discovery between agents and tool providers
- The **client-server architecture** lets agents discover capabilities at runtime
- MCP enables **extensible, decoupled agent systems** where tools can be added without code changes
- Microsoft Agent Framework provides **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
